# L15a: Autotrader Diagnostics, Risk Controls, and Validation
In this lecture, we evaluate an adaptive trading system as an engineered decision process: parameters may drift, performance must be compared with declared baselines, and deployment must be conditional on explicit validation gates.

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
>
> * __Update a Single Index Model online:__ Use exponentially weighted least squares to revise SIM parameters while controlling the effective memory of the estimator.
> * __Diagnose strategy behavior:__ Compare frozen and adaptive configurations using paired paths, drawdown, failure rate, turnover, tail loss, and net present value.
> * __Apply deployment gates:__ Convert quantitative limits into a transparent pass/fail validation report and a pre-approved risk-control configuration.

Let's determine whether the Autotrader is ready to operate—and under what limits.
___


## Examples
The validation workflow will use two coordinated examples:

> [▶ Replay a changing-beta SIM with EWLS](./CHEME-5660-L15a-Example-EWLS-Replay-Fall-2026.ipynb). This example compares frozen and online parameter estimates and diagnoses the stability–responsiveness tradeoff controlled by estimator half-life.
>
> [▶ Build an Autotrader validation report](./CHEME-5660-L15a-Example-Validation-Gates-Fall-2026.ipynb). This example constructs paired validation evidence, evaluates explicit pass/fail gates, and produces a compact risk-control configuration.

The examples separate model adaptation from the governance decision to permit deployment.
___


## Notation
We use the course conventions established in the SIM and portfolio units:

> * $g_{m,t}$ and $g_{i,t}$ denote the market and asset continuously compounded growth rates at observation $t$.
> * $\alpha_i$, $\beta_i$, and $\sigma_{\varepsilon,i}$ denote the SIM intercept, market exposure, and residual standard deviation for asset $i$.
> * $\widehat{\boldsymbol\theta}_{i,t}=(\widehat\alpha_{i,t},\widehat\beta_{i,t})$ is the online parameter estimate.
> * $h$ is the estimator half-life and $\lambda=2^{-1/h}$ is its forgetting factor.
> * $W_{p,T}$ and $W_{b,T}$ denote terminal strategy and benchmark wealth on validation path $p$.

The half-life and the validation thresholds are model inputs and must be reported with the results.
___


## Concept Review: From Adaptive Allocation to Adaptive Estimation
L13a allowed portfolio preferences and target weights to respond to a market-state signal, but the SIM parameters feeding that process remained fixed. A frozen calibration is reasonable only while the relationship between each asset and the market remains sufficiently stable.

Online estimation adds a second adaptation layer. It can improve responsiveness when exposures change, but it also introduces estimator noise, half-life sensitivity, and additional ways for the strategy to overreact. Therefore, the online and frozen configurations must be compared on common paths under the same costs and risk limits.

The engineering question is not simply whether the adaptive estimate moves. It is whether that movement improves the declared portfolio objectives without violating the operating envelope.
___


## Why Frozen Parameters Can Fail
Suppose an asset follows the Single Index Model
$$
g_{i,t}=\alpha_{i,t}+\beta_{i,t}g_{m,t}+\varepsilon_{i,t}.
$$
A batch estimate treats $\alpha_i$ and $\beta_i$ as constants over the calibration window and then carries those values forward. If the firm's business mix, leverage, sector exposure, or market regime changes, the historical estimate may become stale.

Exponentially weighted least squares discounts older observations rather than treating the complete history equally. The half-life controls the tradeoff: short memory responds quickly but produces noisier estimates; long memory is stable but slow to recognize change.

> [▶ Replay a changing-beta SIM with EWLS](./CHEME-5660-L15a-Example-EWLS-Replay-Fall-2026.ipynb). The example makes this stability–responsiveness tradeoff visible on a controlled structural change.
___


## EWLS: Online Single Index Model (SIM) Parameter Estimation

The Single Index Model (SIM) regression $g_{i} = \alpha_{i} + \beta_{i} \, g_{\mathrm{mkt}} + \varepsilon_{i}$ can be re-estimated online as new data arrives. Exponentially weighted least squares (EWLS) discounts old observations with a decay factor so the estimates track regime shifts without a sliding window or batch re-calibration.

> __EWLS Recursion (Theorem):__
>
> Fix asset $i$. Let $\mathbf{x}_{s} = [1,\; g_{\mathrm{mkt},s}]^{\top}$, $y_{s} = g_{i,s}$, and $\boldsymbol{\theta}_{i} = [\alpha_{i},\; \beta_{i}]^{\top}$. Choose a half-life $h$ in trading days and set the decay factor $\delta = 2^{-1/h} \in (0, 1)$. The EWLS estimate at time $t$, defined as the minimizer of $L_{t}(\boldsymbol{\theta}) = \sum_{s=1}^{t} \delta^{t-s}\,(y_{s} - \mathbf{x}_{s}^{\top}\boldsymbol{\theta})^{2}$, is recovered from three running moments updated each step:
>
> $$\mathbf{A}_{t} \leftarrow \delta\,\mathbf{A}_{t-1} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top}, \qquad \mathbf{b}_{t} \leftarrow \delta\,\mathbf{b}_{t-1} + \mathbf{x}_{t} y_{t}, \qquad c_{t} \leftarrow \delta\,c_{t-1} + y_{t}^{2}$$
>
> with the parameter estimate and residual scale given by:
>
> $$\hat{\boldsymbol{\theta}}_{i,t} = \begin{bmatrix} \hat{\alpha}_{i,t} \\ \hat{\beta}_{i,t} \end{bmatrix} = \mathbf{A}_{t}^{-1}\,\mathbf{b}_{t}, \qquad \hat{\sigma}_{\varepsilon,i,t} = \sqrt{\frac{c_{t} - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\,\mathbf{b}_{t}}{[\mathbf{A}_{t}]_{1,1}}}\quad\blacksquare$$
>
> The matrix $\mathbf{A}_{t} \in \mathbb{R}^{2 \times 2}$, vector $\mathbf{b}_{t} \in \mathbb{R}^{2}$, and scalar $c_{t} \in \mathbb{R}$ are sufficient statistics: the entire weighted history collapses into these three running quantities, updated by a single decayed-plus-rank-one step per day. The full proof is in the companion [EWLS recursion derivation notebook](advanced/online_learning/CHEME-5660-L15a-Advanced-EWLS-Recursion-Fall-2026.ipynb).

The recursion runs on every asset in parallel and consumes constant memory and constant compute per trading day. We translate it directly into pseudocode below.

### Algorithm: EWLS SIM Update (Online)

__Initialize__: Given a calibrated prior $(\boldsymbol{\theta}_{i,0}, \sigma_{\varepsilon,i,0})$ for each asset $i$, a prior weight $W_{0} \geq 0$, a half-life $h$ in trading days with decay $\delta = 2^{-1/h}$, and a stream of observations $\{(\mathbf{x}_{t}, y_{t})\}_{t=1}^{T}$ with $\mathbf{x}_{t} = [1,\; g_{\mathrm{mkt},t}]^{\top}$ and $y_{t} = g_{i,t}$, seed the running moments so that $W_{0}$ pseudo-observations carry the prior:
$$\mathbf{A}_{0} \gets W_{0} \cdot \mathbb{E}[\mathbf{x}\mathbf{x}^{\top}], \qquad \mathbf{b}_{0} \gets \mathbf{A}_{0}\,\boldsymbol{\theta}_{i,0}, \qquad c_{0} \gets \boldsymbol{\theta}_{i,0}^{\top}\mathbf{b}_{0} + W_{0}\,\sigma_{\varepsilon,i,0}^{2}$$

For $t = 1, \ldots, T$ __do__:

1. Age the existing moments by the decay factor: $\mathbf{A}_{t} \gets \delta\,\mathbf{A}_{t-1}$, $\mathbf{b}_{t} \gets \delta\,\mathbf{b}_{t-1}$, $c_{t} \gets \delta\,c_{t-1}$.
2. Fold in today's observation $(\mathbf{x}_{t}, y_{t})$:
    $$\mathbf{A}_{t} \gets \mathbf{A}_{t} + \mathbf{x}_{t}\mathbf{x}_{t}^{\top}, \qquad \mathbf{b}_{t} \gets \mathbf{b}_{t} + \mathbf{x}_{t} y_{t}, \qquad c_{t} \gets c_{t} + y_{t}^{2}$$
3. Solve the $2 \times 2$ weighted normal equations for the parameter estimate: $\hat{\boldsymbol{\theta}}_{i,t} \gets \mathbf{A}_{t}^{-1}\,\mathbf{b}_{t}$.
4. Compute the residual standard deviation estimate: $\hat{\sigma}_{\varepsilon,i,t} \gets \sqrt{(c_{t} - \hat{\boldsymbol{\theta}}_{i,t}^{\top}\,\mathbf{b}_{t}) / [\mathbf{A}_{t}]_{1,1}}$.
5. Hand $(\hat{\alpha}_{i,t}, \hat{\beta}_{i,t}, \hat{\sigma}_{\varepsilon,i,t})$ to the engine for use at the next rebalance.

__Output__: Return the running SIM estimates $\{(\hat{\alpha}_{i,t}, \hat{\beta}_{i,t}, \hat{\sigma}_{\varepsilon,i,t})\}_{t=1}^{T}$ and the final sufficient statistics $(\mathbf{A}_{T}, \mathbf{b}_{T}, c_{T})$.

The recursion runs once per asset, in parallel across the universe; only the prior-weighting matrix $\mathbf{A}_{0}$ and the decay factor $\delta$ are shared across assets.

### Prior Seeding and Half-Life

Two knobs control how aggressively EWLS adapts: the prior weight $W_{0}$ that anchors the recursion before any new data arrives, and the half-life $h$ that sets how fast old data fades.

* __Prior weight $W_{0}$:__ Before any observation is processed, the sufficient statistics are seeded as if $W_{0}$ pseudo-observations consistent with the calibrated parameters $\boldsymbol{\theta}_{i,0} = [\alpha_{i,0},\; \beta_{i,0}]^{\top}$ and residual scale $\sigma_{\varepsilon,i,0}$ had been seen, giving $\mathbf{A}_{0} \approx W_{0} \cdot \mathbb{E}[\mathbf{x}\mathbf{x}^{\top}]$, $\mathbf{b}_{0} \approx W_{0} \cdot \mathbb{E}[\mathbf{x}\mathbf{x}^{\top}]\,\boldsymbol{\theta}_{i,0}$, and $c_{0}$ accordingly. Large $W_{0}$ keeps the calibrated values dominant longer; small $W_{0}$ lets the first few real observations move the estimates appreciably.
* __Half-life $h$ (speed):__ A short half-life such as $h = 21$ days (approximately one trading month) puts most of the weight on the last few weeks of data, so the estimates track regime shifts quickly but inherit single-day noise. The $h = 21$ setting is a good choice when responsiveness matters more than smoothness.
* __Half-life $h$ (stability):__ A long half-life such as $h = 126$ days (six months) averages over a much wider window and produces smooth estimates that lag genuine regime transitions. The default $h = 63$ days (one trading quarter) sits between these extremes and is the value the engine ships with for daily rebalancing.

Together $W_{0}$ and $h$ move the estimator along the speed-stability frontier; everything else in the recursion is mechanical.

> __Example__
>
> [▶ Let's replay the engine with EWLS parameter updates](./CHEME-5660-L15a-Example-EWLS-Replay-Fall-2026.ipynb). We run the engine on the the adaptive-allocation unit Monte Carlo ensemble with frozen vs. online parameters, compare distributional metrics, and zoom into single paths to see what one realized future looks like.

___

## Validation and Compliance

Before the engine goes to production in the production-operations lecture, it must pass a formal validation. The validation report applies five pass/fail gates to the Monte Carlo backtest results:

| Criterion | Threshold | Rationale |
|-----------|-----------|-----------|
| **Median Sharpe** $\geq$ **0.3** | Risk-adjusted growth must exceed a low bar | Below 0.3, the strategy is not compensating for the risk it takes |
| **Median Max Drawdown** $\leq$ **25%** | Worst-case capital loss must be bounded | Larger drawdowns trigger investor redemptions and regulatory scrutiny |
| **Failure Rate** $\leq$ **10%** | At most 10% of paths end below initial capital | Systematic losses indicate a flawed strategy |
| **Wealth versus Frozen Baseline** $\geq$ **1.0** | Median terminal wealth must beat the frozen-parameter engine | An online learner that does not beat its frozen ancestor is not earning its complexity |
| **Median NPV** $\geq$ **0** | Median path must beat the risk-free baseline in present-value terms | A strategy whose median NPV is negative is dominated by a T-bill |

A strategy that passes all five gates across model-generated paths has demonstrated robustness to regime shifts, fat tails, and volatility clustering. It is not guaranteed to work in production, but it has cleared the minimum bar for deployment.

The validation report also produces a set of **compliance parameters** that become the pre-approved operating limits for the production-operations lecture's production system:

* **Concentration cap:** maximum portfolio weight in any single asset (e.g., 40%)
* **Drawdown gate:** maximum drawdown before the engine de-risks to cash (e.g., 15%)
* **Turnover limit:** maximum daily trade volume as a fraction of portfolio value (e.g., 50%)
* **Position size limit:** maximum dollar exposure per individual trade (e.g., $5,000)

In the production-operations lecture, trades within these limits be eligible for automated execution; trades that exceed them are queued for human review. The compliance configuration is the bridge between _validated in simulation_ and _approved for production._

> __Example__
>
> [▶ Let's build the validation report and export compliance config](./CHEME-5660-L15a-Example-Validation-Gates-Fall-2026.ipynb). We evaluate the EWLS engine against the frozen Cobb-Douglas baseline across the five gates, produce pass/fail verdicts, and export the compliance parameters for the production-operations lecture.

___

## Summary
This lecture treated model adaptation and deployment authorization as separate engineering decisions.

> __Key Takeaways:__
>
> * __EWLS makes estimator memory explicit:__ The half-life controls how rapidly old observations lose influence.
> * __Adaptive and frozen models require paired evaluation:__ Common scenarios isolate policy differences from path differences.
> * __Deployment gates must be declared and auditable:__ Reward, drawdown, failure, benchmark, NPV, turnover, and operational limits each test a different failure mode.

Passing the classroom gates authorizes the captured production exercise only; it does not authorize unattended live trading.
___


## Disclaimer and Risks
This material is for educational purposes only and does not constitute investment advice. Passing a simplified validation gate does not establish that a model is suitable for live deployment.
